In [ ]:
import numpy as np
import ssqpy
from time import time, sleep

np.set_printoptions(suppress=True)

# Build MPC
ssqpy.setSilentMode()

dt = 0.01
MPC_H = 15

V_WGT = 3e-2
U_WGT = 2e-3
TIP_WGT = 4.2
J1_WGT = 0.0
J2_WGT = 2.0

torque_clip = 0.1

model = ssqpy.model.Model(
    MPC_H,
    dt,
    urdf_path="pendubot.urdf",
    actuated_joints=[0],
    solver_mode=ssqpy.model.SolverMode.InverseDynamics
)

nq = model.getnq()
nv = model.getnv()
nu = model.getnu()

vel_cost = ssqpy.model.costs.SquaredJointVelocityCost(model, V_WGT)
u_cost = ssqpy.model.costs.SquaredControlCost(model, U_WGT)
tip_cost = ssqpy.model.costs.FrameSquaredTranslationErrorCost(
    model, "tip", np.array((0.0, 0.0, 0.2)), TIP_WGT
)
config_cost = ssqpy.model.costs.SquaredConfigurationErrorCost(
    model, np.array((np.pi, 0.0)), np.array((J1_WGT, J2_WGT))
)

for k in range(MPC_H):
    model.addCost(k, vel_cost)
    model.addCost(k, u_cost)
    model.addCost(k, tip_cost)
    model.addCost(k, config_cost)

model.addCost(MPC_H, vel_cost)
model.addCost(MPC_H, tip_cost)
model.addCost(MPC_H, config_cost)

ssqp_params = ssqpy.solvers.ssqpParams()
ssqp_params.tolerance = 1e-1

admm_params = ssqpy.solvers.admmParams()
admm_params.abs_tolerance = 1e-2
admm_params.rel_tolerance = 1e-2
admm_params.warm_start = True
ssqp_params.admmParams = admm_params


mpc = ssqpy.solvers.MPC(
    model, ssqp_params, sqp_iters=4, qp_iters=100
)

In [ ]:
def wrap_to_reference(theta, reference=np.pi):
    return reference + np.arctan2(
        np.sin(theta - reference), np.cos(theta - reference)
    )

t0 = time()
timestamps = []
frequencies = []
data = []

In [ ]:
from cloudpendulumclient.client import Client

user_token = "MY_TOKEN"

Tf = 15.0
client = Client()
session_token, livestream_url = client.start_experiment(
    user_token = user_token,
    experiment_type = "Pendubot",
    experiment_time = Tf,
    preparation_time = 5.0,
    record = True
)

print(client.get_cells(user_token))

print("Received response from server!")
print("Session token: ", session_token)
print("Livestream url: ", livestream_url)

current_time = 0.0

np.set_printoptions(suppress=True)

start_all = time()
while (time() - start_all) < Tf:
    start = time()

    mq = client.get_position(session_token)
    mv = client.get_velocity(session_token)
    mt = client.get_torque(session_token)

    try:
        u = mpc.step(np.hstack((mq, mv)))[1].stage(0)
    except RuntimeError:
        u = np.zeros(nv)

    tau = model.inverseDynamics(np.array(mq), np.array(mv), u)
    tau = np.clip(tau, -torque_clip, torque_clip)

    client.set_torque(tau[:1], session_token)

    elapsed = time() - start

    #print(f"Frequency: {1/elapsed:.2f}Hz")
    timestamps.append(time() - t0)
    data.append(
        (
            np.array(
                (wrap_to_reference(mq[0]), wrap_to_reference(mq[1], 0.0))
            ),
            mv,
            mt,
        )
    )
    frequencies.append(1 / elapsed)
        
url = client.stop_experiment(session_token)

print("Final state:", mq)

In [ ]:
import matplotlib.pyplot as plt
# Unpack data
positions = np.array([item[0] for item in data])  # (N, 2)
velocities = np.array([item[1] for item in data])  # (N, 2)
torques = np.array([item[2] for item in data])[:, 0]  # (N, 2)
frequencies = np.array(frequencies)

fig = plt.figure(figsize=(10, 8))
gs = fig.add_gridspec(4, 1, height_ratios=[1, 1, 1, 1])

ax_pos = fig.add_subplot(gs[0, 0])
ax_vel = fig.add_subplot(gs[1, 0], sharex=ax_pos)
ax_tau = fig.add_subplot(gs[2, 0], sharex=ax_pos)
ax_freq = fig.add_subplot(gs[3, 0], sharex=ax_pos)

t = np.array(timestamps)

# --- Position ---
for i, label in enumerate(["q₀", "q₁"]):
    ax_pos.plot(t, positions[:, i], label=label)

ax_pos.set_title("Position")
ax_pos.set_ylabel("Position")
ax_pos.legend()
ax_pos.grid(True)

# --- Velocity ---
for i, label in enumerate(["v₀", "v₁"]):
    ax_vel.plot(t, velocities[:, i], label=label)

ax_vel.set_title("Velocity")
ax_vel.set_ylabel("Velocity")
ax_vel.legend()
ax_vel.grid(True)

# --- Torque (τ₀) ---
ax_tau.plot(t, torques, label="τ₀")

ax_tau.set_title("Torque")
ax_tau.set_ylabel("τ₀")
ax_tau.grid(True)

# --- Frequency ---
ax_freq.plot(t, frequencies, label=f"f")

ax_freq.set_title("Control Frequency")
ax_freq.set_xlabel("Time (s)")
ax_freq.set_ylabel("Frequency (Hz)")
ax_freq.set_yscale("log")

ax_freq.legend()
ax_freq.grid(True)

plt.tight_layout()
plt.savefig(f"pendubot.pdf")
plt.show()

In [ ]:
import urllib.request
urllib.request.urlretrieve(url, "pendubot.flv") 